# 1.Buisness Understanding

## 2.Load and EDA

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [11]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


(7043, 21)

In [9]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

identifier_columns = [
    col for col in df.columns
    if col.lower().endswith('id')
]

categorical_columns = [
    col for col in df.select_dtypes(include=['object', 'category', 'bool']).columns
    if col not in identifier_columns
]

# Treat binary indicators as categorical rather than continuous numeric features.
categorical_columns += [
    col for col in df.select_dtypes(include='number').columns
    if col not in identifier_columns and df[col].nunique(dropna=True) <= 2
]

numerical_columns = [
    col for col in df.select_dtypes(include='number').columns
    if col not in identifier_columns and col not in categorical_columns
]

print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)
print("Identifier columns:", identifier_columns)

Numerical columns: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn', 'SeniorCitizen']
Identifier columns: ['customerID']


In [10]:
# Data types
print("Data types:")
display(df.dtypes.to_frame("Data type"))

# Missing values
missing = df.isna().sum()
missing_summary = pd.DataFrame({
    "Missing values": missing,
    "Missing (%)": (missing / len(df) * 100).round(2)
})
print("Missing values:")
display(missing_summary[missing_summary["Missing values"] > 0])

# Duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

# Target distribution
print("Churn distribution:")
display(pd.DataFrame({
    "Count": df["Churn"].value_counts(),
    "Percentage": (df["Churn"].value_counts(normalize=True) * 100).round(2)
}))

Data types:


,Data type
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


Missing values:


,Missing values,Missing (%)
TotalCharges,11,0.16


Duplicate rows: 0
Churn distribution:


,Count,Percentage
Churn,,
No,5174,73.46
Yes,1869,26.54


In [ ]:
# Churn distribution and relationships with key customer attributes
eda_df = df.copy()
eda_df["ChurnFlag"] = eda_df["Churn"].map({"No": 0, "Yes": 1})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Overall churn counts
churn_counts = eda_df["Churn"].value_counts().reindex(["No", "Yes"])
axes[0, 0].bar(churn_counts.index, churn_counts.values, color=["steelblue", "tomato"])
axes[0, 0].set(title="Customer Churn Counts", xlabel="Churn", ylabel="Number of customers")

# 2. Churn rate by contract type
contract_churn = (
    eda_df.groupby("Contract", observed=False)["ChurnFlag"]
    .mean()
    .mul(100)
    .sort_values()
)
axes[0, 1].bar(contract_churn.index, contract_churn.values, color="darkorange")
axes[0, 1].set(title="Churn Rate by Contract Type", xlabel="Contract", ylabel="Churn rate (%)")
axes[0, 1].tick_params(axis="x", rotation=15)

# 3. Tenure by churn status
sns.boxplot(data=eda_df, x="Churn", y="tenure", order=["No", "Yes"], ax=axes[1, 0])
axes[1, 0].set(title="Customer Tenure by Churn Status", xlabel="Churn", ylabel="Tenure (months)")

# 4. Monthly charges by churn status
sns.boxplot(data=eda_df, x="Churn", y="MonthlyCharges", order=["No", "Yes"], ax=axes[1, 1])
axes[1, 1].set(title="Monthly Charges by Churn Status", xlabel="Churn", ylabel="Monthly charges")

plt.tight_layout()
plt.show()